In [1]:
# !!!True for FINALIZE i.e. GENERATE SUBMISSION!!!
# !!!set MANUALLY!!!
finalize: bool = True

# SETUP

In [2]:
#Hyperparameters
CFG = {
    'dropout':         0.3,
    'lr':              1e-3,
    'weight_decay':    1e-4,
    'epochs':          150,
    'patience':        20,    # early stopping
    'val_season':      2021,
    'label_smoothing': 0.05,  # avoids overconfident predictions (helps log loss)
    'finalize': finalize,
    'seed': 923434 #deterministic seed for comparison purposes
}

print(f"Config: {CFG}")

Config: {'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001, 'epochs': 150, 'patience': 20, 'val_season': 2021, 'label_smoothing': 0.05, 'finalize': True, 'seed': 923434}


In [3]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_info_columns', 999)
#pd.set_option('display.max_info_rows', 999)


torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

#DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE = torch.accelerator.current_accelerator()
print(f'DEVICE::{DEVICE}')
if DEVICE.type in ('xpu', 'cpu'):
    old_epochs = CFG['epochs']
    CFG['epochs'] = 2 #for testing if further code works locally; actual training on kaggle gpus
    print(f"Low compute - proceeding in no-training mode. Current training epochs = {CFG['epochs']} (was {old_epochs})")

DEVICE::cuda


In [4]:
import os
# Check if we are running on Kaggle's servers
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    DATA_DIR = '/kaggle/input/competitions/march-machine-learning-mania-2026'
else:
    DATA_DIR = '.'

# Load Data

In [5]:
def load_data(data_dir: str) -> dict:
    """Load all competition CSV files."""
    files = {
        'm_teams':   'MTeams.csv',
        'w_teams':   'WTeams.csv',
        'm_regular_detailed': 'MRegularSeasonDetailedResults.csv',
        'w_regular_detailed': 'WRegularSeasonDetailedResults.csv',
        'm_tourney_detailed': 'MNCAATourneyDetailedResults.csv',
        'w_tourney_detailed': 'WNCAATourneyDetailedResults.csv',
        'm_tourney': 'MNCAATourneyCompactResults.csv',
        'w_tourney': 'WNCAATourneyCompactResults.csv',
        'm_seeds':   'MNCAATourneySeeds.csv',
        'w_seeds':   'WNCAATourneySeeds.csv',
        'sample_sub':'SampleSubmissionStage1.csv',
        'sample_sub2':'SampleSubmissionStage2.csv'
    }

    data = {}
    for key, fname in files.items():
        path = f"{data_dir}/{fname}"
        try:
            data[key] = pd.read_csv(path)
            print(f"{fname}: {data[key].shape}")
        except FileNotFoundError:
            print(f"{fname} not found Ă˘â‚¬â€ť skipping")
            data[key] = pd.DataFrame()

    return data

print("Loading data...")
DATA = load_data(DATA_DIR)

Loading data...
MTeams.csv: (381, 4)
WTeams.csv: (379, 2)
MRegularSeasonDetailedResults.csv: (124529, 34)
WRegularSeasonDetailedResults.csv: (87187, 34)
MNCAATourneyDetailedResults.csv: (1449, 34)
WNCAATourneyDetailedResults.csv: (961, 34)
MNCAATourneyCompactResults.csv: (2585, 8)
WNCAATourneyCompactResults.csv: (1717, 8)
MNCAATourneySeeds.csv: (2694, 3)
WNCAATourneySeeds.csv: (1812, 3)
SampleSubmissionStage1.csv: (519144, 2)
SampleSubmissionStage2.csv: (132133, 2)


# Data preparation

In [6]:
DATA['m_regular_detailed']

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124524,2026,132,1335,88,1463,84,N,1,29,64,14,28,16,16,11,26,17,13,5,4,16,31,71,9,24,13,17,17,24,23,9,9,4,16
124525,2026,132,1345,80,1276,72,N,0,30,57,4,14,16,22,7,22,21,2,4,5,13,30,64,7,24,5,6,11,22,15,7,2,3,18
124526,2026,132,1378,70,1455,55,N,0,23,61,6,22,18,23,14,27,13,10,10,3,20,19,56,4,19,13,20,12,27,8,14,7,7,19
124527,2026,132,1433,70,1173,62,N,0,23,57,11,24,13,19,6,28,12,5,9,4,18,23,53,7,21,9,17,8,31,16,11,3,4,18


In [7]:
regular_results_detailed: pd.DataFrame = pd.concat([DATA['m_regular_detailed'], DATA['w_regular_detailed']], axis=0, ignore_index=True)
tourney_results_detailed: pd.DataFrame = pd.concat([DATA['m_tourney_detailed'], DATA['w_tourney_detailed']], axis=0, ignore_index=True)

regular_results_detailed


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,8,18,17,29,17,26,15,10,5,2,25,22,73,3,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,3,9,17,31,6,19,11,12,14,2,18,18,49,6,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,6,14,11,13,17,22,12,14,4,4,20,24,62,6,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211711,2026,131,3471,60,3218,48,N,0,25,69,3,16,7,8,8,27,13,7,11,1,9,18,48,4,16,8,8,2,31,9,17,3,6,15
211712,2026,132,3158,68,3220,56,N,0,23,61,9,27,13,21,16,26,12,9,5,1,12,23,58,7,21,3,5,9,25,10,13,2,4,17
211713,2026,132,3192,79,3254,57,H,0,31,60,10,19,7,11,9,32,15,10,4,1,18,19,55,7,20,12,14,5,19,2,11,5,4,15
211714,2026,132,3221,77,3250,70,H,0,31,60,5,17,10,12,13,23,16,12,3,4,10,26,55,9,24,9,12,8,17,17,11,6,2,12


In [8]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211716 entries, 0 to 211715
Data columns (total 34 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Season   211716 non-null  int64 
 1   DayNum   211716 non-null  int64 
 2   WTeamID  211716 non-null  int64 
 3   WScore   211716 non-null  int64 
 4   LTeamID  211716 non-null  int64 
 5   LScore   211716 non-null  int64 
 6   WLoc     211716 non-null  object
 7   NumOT    211716 non-null  int64 
 8   WFGM     211716 non-null  int64 
 9   WFGA     211716 non-null  int64 
 10  WFGM3    211716 non-null  int64 
 11  WFGA3    211716 non-null  int64 
 12  WFTM     211716 non-null  int64 
 13  WFTA     211716 non-null  int64 
 14  WOR      211716 non-null  int64 
 15  WDR      211716 non-null  int64 
 16  WAst     211716 non-null  int64 
 17  WTO      211716 non-null  int64 
 18  WStl     211716 non-null  int64 
 19  WBlk     211716 non-null  int64 
 20  WPF      211716 non-null  int64 
 21  LFGM     2

In [9]:
# double the dataset with swapped team positions in box scores
def prepare_data(df: pd.DataFrame):
    df["LLoc"] = [{"H": "A", "A": "H", "N":"N"}[item] for item in df["WLoc"]]

    df = df[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
            "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF", "LLoc",
            "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF", "WLoc"]]
    
    
    """
    # adjustment factor for overtimes, as more stats are accumulated during overtimes
    adjot = (40 + 5 * df["NumOT"]) / 40
    adjcols = ["LScore", "WScore", 
               "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
               "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]
    for col in adjcols:
        df[col] = df[col] / adjot
    """  
    #mapp = {'W': 'T1_', 'L': 'T2_'}  
    def label_swap(lab: str, mapp: dict) -> str:
        first = lab[0]
        if first in mapp:
            return mapp[first] + lab[1:]
        return lab
    
    #df.drop("WLoc", axis=1)
    dfswap = df.copy()
    df.columns = [label_swap(x, {'W': 'T1_', 'L': 'T2_'}) for x in list(df.columns)]
    dfswap.columns = [label_swap(x, {'L': 'T1_', 'W': 'T2_'}) for x in list(dfswap.columns)]
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output["PointDiff"] = output["T1_Score"] - output["T2_Score"]
    output["win"] = (output["PointDiff"] > 0) * 1
    output["men_women"] = (output["T1_TeamID"].apply(lambda t: str(t).startswith("1"))) * 1  # 0: women, 1: men
    return output

regular_results_detailed = prepare_data(regular_results_detailed)
tourney_results_detailed = prepare_data(tourney_results_detailed)

regular_results_detailed

,Season,DayNum,T2_TeamID,T2_Score,T1_TeamID,T1_Score,NumOT,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_Loc,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_Loc,PointDiff,win,men_women
0,2003,10,1328,62,1104,68,0,22,53,2,10,16,22,10,22,8,18,9,2,20,N,27,58,3,14,11,18,14,24,13,23,7,1,22,N,6,1,1
1,2003,10,1393,63,1272,70,0,24,67,6,24,9,20,20,25,7,12,8,6,16,N,26,62,8,20,10,19,15,28,16,13,4,4,18,N,7,1,1
2,2003,11,1437,61,1266,73,0,22,73,3,26,14,23,31,22,9,12,2,5,23,N,24,58,8,18,17,29,17,26,15,10,5,2,25,N,12,1,1
3,2003,11,1457,50,1296,56,0,18,49,6,22,8,15,17,20,9,19,4,3,23,N,18,38,3,9,17,31,6,19,11,12,14,2,18,N,6,1,1
4,2003,11,1208,71,1400,77,0,24,62,6,16,17,27,21,15,12,10,7,1,14,N,30,61,6,14,11,13,17,22,12,14,4,4,20,N,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423427,2026,131,3471,60,3218,48,0,25,69,3,16,7,8,8,27,13,7,11,1,9,N,18,48,4,16,8,8,2,31,9,17,3,6,15,N,-12,0,0
423428,2026,132,3158,68,3220,56,0,23,61,9,27,13,21,16,26,12,9,5,1,12,N,23,58,7,21,3,5,9,25,10,13,2,4,17,N,-12,0,0
423429,2026,132,3192,79,3254,57,0,31,60,10,19,7,11,9,32,15,10,4,1,18,H,19,55,7,20,12,14,5,19,2,11,5,4,15,A,-22,0,0
423430,2026,132,3221,77,3250,70,0,31,60,5,17,10,12,13,23,16,12,3,4,10,H,26,55,9,24,9,12,8,17,17,11,6,2,12,A,-7,0,0


In [10]:
regular_results_detailed.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423432 entries, 0 to 423431
Data columns (total 38 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Season     423432 non-null  int64 
 1   DayNum     423432 non-null  int64 
 2   T2_TeamID  423432 non-null  int64 
 3   T2_Score   423432 non-null  int64 
 4   T1_TeamID  423432 non-null  int64 
 5   T1_Score   423432 non-null  int64 
 6   NumOT      423432 non-null  int64 
 7   T2_FGM     423432 non-null  int64 
 8   T2_FGA     423432 non-null  int64 
 9   T2_FGM3    423432 non-null  int64 
 10  T2_FGA3    423432 non-null  int64 
 11  T2_FTM     423432 non-null  int64 
 12  T2_FTA     423432 non-null  int64 
 13  T2_OR      423432 non-null  int64 
 14  T2_DR      423432 non-null  int64 
 15  T2_Ast     423432 non-null  int64 
 16  T2_TO      423432 non-null  int64 
 17  T2_Stl     423432 non-null  int64 
 18  T2_Blk     423432 non-null  int64 
 19  T2_PF      423432 non-null  int64 
 20  T2_L

In [11]:
seeds: pd.DataFrame = pd.concat([DATA['m_seeds'], DATA['w_seeds']], axis=0, ignore_index=True)
seeds

,Season,Seed,TeamID
0,1985,W01,1207
1,1985,W02,1210
2,1985,W03,1228
3,1985,W04,1260
4,1985,W05,1374
...,...,...,...
4501,2026,Z12,3211
4502,2026,Z13,3453
4503,2026,Z14,3158
4504,2026,Z15,3239


In [12]:
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))
seeds

,Season,Seed,TeamID,seed
0,1985,W01,1207,1
1,1985,W02,1210,2
2,1985,W03,1228,3
3,1985,W04,1260,4
4,1985,W05,1374,5
...,...,...,...,...
4501,2026,Z12,3211,12
4502,2026,Z13,3453,13
4503,2026,Z14,3158,14
4504,2026,Z15,3239,15


In [13]:
def seed_transform(df: pd.DataFrame) -> pd.DataFrame:
    seeds_T1 = seeds[["Season", "TeamID", "seed"]].copy()
    seeds_T2 = seeds[["Season", "TeamID", "seed"]].copy()
    seeds_T1.columns = ["Season", "T1_TeamID", "T1_seed"]
    seeds_T2.columns = ["Season", "T2_TeamID", "T2_seed"]

    df = pd.merge(df, seeds_T1, on=["Season", "T1_TeamID"], how="left")
    df = pd.merge(df, seeds_T2, on=["Season", "T2_TeamID"], how="left")

    # --- THE FIX ---
    # Assign a "ghost seed" to teams that didn't make the tournament
    # Seed 20 ensures they are mathematically treated as weaker than a 16-seed
    UNSEEDED_VAL = 20
    df["T1_seed"] = df["T1_seed"].fillna(UNSEEDED_VAL)
    df["T2_seed"] = df["T2_seed"].fillna(UNSEEDED_VAL)
    
    df["Seed_diff"] = df["T2_seed"] - df["T1_seed"]

    return df

tourney_results_detailed = seed_transform(tourney_results_detailed)
regular_results_detailed = seed_transform(regular_results_detailed)

regular_results_detailed

,Season,DayNum,T2_TeamID,T2_Score,T1_TeamID,T1_Score,NumOT,T2_FGM,T2_FGA,T2_FGM3,T2_FGA3,T2_FTM,T2_FTA,T2_OR,T2_DR,T2_Ast,T2_TO,T2_Stl,T2_Blk,T2_PF,T2_Loc,T1_FGM,T1_FGA,T1_FGM3,T1_FGA3,T1_FTM,T1_FTA,T1_OR,T1_DR,T1_Ast,T1_TO,T1_Stl,T1_Blk,T1_PF,T1_Loc,PointDiff,win,men_women,T1_seed,T2_seed,Seed_diff
0,2003,10,1328,62,1104,68,0,22,53,2,10,16,22,10,22,8,18,9,2,20,N,27,58,3,14,11,18,14,24,13,23,7,1,22,N,6,1,1,10.0,1.0,-9.0
1,2003,10,1393,63,1272,70,0,24,67,6,24,9,20,20,25,7,12,8,6,16,N,26,62,8,20,10,19,15,28,16,13,4,4,18,N,7,1,1,7.0,3.0,-4.0
2,2003,11,1437,61,1266,73,0,22,73,3,26,14,23,31,22,9,12,2,5,23,N,24,58,8,18,17,29,17,26,15,10,5,2,25,N,12,1,1,3.0,20.0,17.0
3,2003,11,1457,50,1296,56,0,18,49,6,22,8,15,17,20,9,19,4,3,23,N,18,38,3,9,17,31,6,19,11,12,14,2,18,N,6,1,1,20.0,20.0,0.0
4,2003,11,1208,71,1400,77,0,24,62,6,16,17,27,21,15,12,10,7,1,14,N,30,61,6,14,11,13,17,22,12,14,4,4,20,N,6,1,1,1.0,20.0,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423427,2026,131,3471,60,3218,48,0,25,69,3,16,7,8,8,27,13,7,11,1,9,N,18,48,4,16,8,8,2,31,9,17,3,6,15,N,-12,0,0,20.0,14.0,-6.0
423428,2026,132,3158,68,3220,56,0,23,61,9,27,13,21,16,26,12,9,5,1,12,N,23,58,7,21,3,5,9,25,10,13,2,4,17,N,-12,0,0,20.0,14.0,-6.0
423429,2026,132,3192,79,3254,57,0,31,60,10,19,7,11,9,32,15,10,4,1,18,H,19,55,7,20,12,14,5,19,2,11,5,4,15,A,-22,0,0,20.0,15.0,-5.0
423430,2026,132,3221,77,3250,70,0,31,60,5,17,10,12,13,23,16,12,3,4,10,H,26,55,9,24,9,12,8,17,17,11,6,2,12,A,-7,0,0,20.0,15.0,-5.0


In [14]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423432 entries, 0 to 423431
Data columns (total 41 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Season     423432 non-null  int64  
 1   DayNum     423432 non-null  int64  
 2   T2_TeamID  423432 non-null  int64  
 3   T2_Score   423432 non-null  int64  
 4   T1_TeamID  423432 non-null  int64  
 5   T1_Score   423432 non-null  int64  
 6   NumOT      423432 non-null  int64  
 7   T2_FGM     423432 non-null  int64  
 8   T2_FGA     423432 non-null  int64  
 9   T2_FGM3    423432 non-null  int64  
 10  T2_FGA3    423432 non-null  int64  
 11  T2_FTM     423432 non-null  int64  
 12  T2_FTA     423432 non-null  int64  
 13  T2_OR      423432 non-null  int64  
 14  T2_DR      423432 non-null  int64  
 15  T2_Ast     423432 non-null  int64  
 16  T2_TO      423432 non-null  int64  
 17  T2_Stl     423432 non-null  int64  
 18  T2_Blk     423432 non-null  int64  
 19  T2_PF      423432 non-n

In [15]:
boxcols = [
    "T1_Score", "T1_FGM", "T1_FGA", "T1_FGM3", "T1_FGA3", "T1_FTM", "T1_FTA",
    "T1_OR", "T1_DR", "T1_Ast", "T1_TO", "T1_Stl", "T1_Blk", "T1_PF",
    "T2_Score", "T2_FGM", "T2_FGA", "T2_FGM3", "T2_FGA3", "T2_FTM", "T2_FTA",
    "T2_OR", "T2_DR", "T2_Ast", "T2_TO", "T2_Stl", "T2_Blk", "T2_PF",
    "PointDiff",
]


In [16]:
def add_rolling_averages(regular_df: pd.DataFrame, tourney_df: pd.DataFrame):
    if 'T1_avg_Score' in regular_df.columns:
        print("Averages already calculated! Skipping...")
        return regular_df, tourney_df
    print("Calculating rolling season averages...")
    
    # 1. CRITICAL: Sort chronologically so the expanding mean works correctly
    regular_df = regular_df.sort_values(by=['Season', 'DayNum']).reset_index(drop=True)
    
    # Identify the stats we want to average
    exclude_cols = ['T1_TeamID', 'T1_Loc']
    stat_cols = [col for col in regular_df.columns if col.startswith('T1_') and col not in exclude_cols]
    
    # ---------------------------------------------------------
    # PART 1: REGULAR SEASON (ROLLING AVERAGES)
    # ---------------------------------------------------------
    
    # 2. Calculate the expanding rolling average and SHIFT BY 1
    avg_cols = []
    for col in stat_cols:
        print(col, end=' ')
        avg_col_name = col.replace('T1_', 'T1_avg_')
        avg_cols.append(avg_col_name)
        
        # Group by team and season, calculate expanding mean, and shift(1) to prevent leakage
        regular_df[avg_col_name] = regular_df.groupby(['Season', 'T1_TeamID'])[col].transform(
            lambda x: x.expanding().mean().shift(1)
        )
        
    # 3. Get T2's rolling averages via a self-merge
    # Because the dataset is doubled, T2's rolling stats going into a game are simply 
    # T1's rolling stats from the exact same game on the other row!
    t2_mapping = regular_df[['Season', 'DayNum', 'T1_TeamID'] + avg_cols].copy()
    t2_mapping.columns = ['Season', 'DayNum', 'T2_TeamID'] + [c.replace('T1_avg_', 'T2_avg_') for c in avg_cols]
    
    # Merge T2 rolling stats back onto the main dataframe
    regular_df = pd.merge(regular_df, t2_mapping, on=['Season', 'DayNum', 'T2_TeamID'], how='left')
    
    # ---------------------------------------------------------
    # PART 2: TOURNAMENT (FINAL AVERAGES)
    # ---------------------------------------------------------
    
    # For the tournament, the season is over, so we just want the FINAL season average.
    # This is exactly what the old function did, so we just calculate the standard mean.
    final_averages = regular_df.groupby(['Season', 'T1_TeamID'])[stat_cols].mean().reset_index()
    
    t1_final = final_averages.copy()
    t1_final.columns = ['Season', 'T1_TeamID'] + avg_cols
    
    t2_final = final_averages.copy()
    t2_final.columns = ['Season', 'T2_TeamID'] + [c.replace('T1_avg_', 'T2_avg_') for c in avg_cols]
    
    tourney_df = pd.merge(tourney_df, t1_final, on=['Season', 'T1_TeamID'], how='left')
    tourney_df = pd.merge(tourney_df, t2_final, on=['Season', 'T2_TeamID'], how='left')
    
    # ---------------------------------------------------------
    # PART 3: CLEANUP
    # ---------------------------------------------------------
    
    # Drop the raw in-game box score stats so they don't leak
    raw_t1_stats = [col for col in stat_cols]
    raw_t2_stats = [col.replace('T1_', 'T2_') for col in stat_cols]
    
    regular_df = regular_df.drop(columns=raw_t1_stats + raw_t2_stats)
    tourney_df = tourney_df.drop(columns=raw_t1_stats + raw_t2_stats)
    
    # The very first game of the season will have NaN for rolling averages (since there's no history)
    # We fill these with 0 so the neural network can still process them
    regular_df = regular_df.fillna(0)
    
    return regular_df, tourney_df

# Execute
regular_results_detailed, tourney_results_detailed = add_rolling_averages(regular_results_detailed, tourney_results_detailed)

Calculating rolling season averages...
T1_Score T1_FGM T1_FGA T1_FGM3 T1_FGA3 T1_FTM T1_FTA T1_OR T1_DR T1_Ast T1_TO T1_Stl T1_Blk T1_PF T1_seed 

In [17]:
regular_results_detailed

,Season,DayNum,T2_TeamID,T1_TeamID,NumOT,T2_Loc,T1_Loc,PointDiff,win,men_women,Seed_diff,T1_avg_Score,T1_avg_FGM,T1_avg_FGA,T1_avg_FGM3,T1_avg_FGA3,T1_avg_FTM,T1_avg_FTA,T1_avg_OR,T1_avg_DR,T1_avg_Ast,T1_avg_TO,T1_avg_Stl,T1_avg_Blk,T1_avg_PF,T1_avg_seed,T2_avg_Score,T2_avg_FGM,T2_avg_FGA,T2_avg_FGM3,T2_avg_FGA3,T2_avg_FTM,T2_avg_FTA,T2_avg_OR,T2_avg_DR,T2_avg_Ast,T2_avg_TO,T2_avg_Stl,T2_avg_Blk,T2_avg_PF,T2_avg_seed
0,2003,10,1328,1104,0,N,N,6,1,1,-9.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
1,2003,10,1393,1272,0,N,N,7,1,1,-4.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
2,2003,10,1104,1328,0,N,N,-6,0,1,9.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
3,2003,10,1272,1393,0,N,N,-7,0,1,4.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
4,2003,11,1437,1266,0,N,N,12,1,1,17.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423429,2026,132,1465,1430,0,A,H,-2,0,1,-7.0,78.241379,28.448276,57.275862,6.448276,19.034483,14.896552,21.689655,9.689655,22.724138,17.551724,13.586207,9.413793,5.275862,19.000000,20.0,73.343750,25.718750,59.343750,6.343750,18.843750,15.562500,21.750000,12.000000,24.000000,10.375000,11.437500,5.437500,2.906250,18.656250,13.0
423430,2026,132,3158,3220,0,N,N,-12,0,0,-6.0,53.580645,20.322581,55.096774,4.967742,16.806452,7.967742,11.741935,9.193548,20.000000,10.741935,14.967742,7.000000,2.129032,16.419355,20.0,71.103448,26.103448,65.793103,7.206897,22.172414,11.689655,15.793103,13.103448,22.206897,12.931034,12.000000,10.379310,1.931034,15.103448,14.0
423431,2026,132,3192,3254,0,H,A,-22,0,0,-5.0,70.137931,24.551724,58.000000,7.206897,20.827586,13.827586,19.000000,9.689655,23.793103,13.517241,16.000000,6.689655,4.103448,17.344828,20.0,67.516129,24.967742,59.645161,8.870968,26.096774,8.709677,12.000000,12.290323,24.225806,13.516129,12.258065,6.193548,3.354839,13.870968,15.0
423432,2026,132,3221,3250,0,H,A,-7,0,0,-5.0,66.266667,22.466667,52.233333,8.500000,25.833333,12.833333,16.600000,6.533333,19.100000,14.700000,14.900000,7.800000,1.500000,14.600000,20.0,60.806452,22.806452,55.741935,5.225806,18.580645,9.967742,13.322581,7.354839,25.387097,14.774194,13.870968,7.677419,3.451613,13.225806,15.0


In [18]:
regular_results_detailed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423434 entries, 0 to 423433
Data columns (total 41 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Season        423434 non-null  int64  
 1   DayNum        423434 non-null  int64  
 2   T2_TeamID     423434 non-null  int64  
 3   T1_TeamID     423434 non-null  int64  
 4   NumOT         423434 non-null  int64  
 5   T2_Loc        423434 non-null  object 
 6   T1_Loc        423434 non-null  object 
 7   PointDiff     423434 non-null  int64  
 8   win           423434 non-null  int64  
 9   men_women     423434 non-null  int64  
 10  Seed_diff     423434 non-null  float64
 11  T1_avg_Score  423434 non-null  float64
 12  T1_avg_FGM    423434 non-null  float64
 13  T1_avg_FGA    423434 non-null  float64
 14  T1_avg_FGM3   423434 non-null  float64
 15  T1_avg_FGA3   423434 non-null  float64
 16  T1_avg_FTM    423434 non-null  float64
 17  T1_avg_FTA    423434 non-null  float64
 18  T1_a

In [19]:
def compute_rolling_elo(regular_df: pd.DataFrame, tourney_df: pd.DataFrame, k: float = 20.0, initial: float = 1500.0):
    """
    Computes rolling (pre-game) Elo ratings for the regular season,
    and end-of-season Elo ratings for the tournament.
    """
    print("Calculating rolling Elo ratings...")
    if 'T1_elo' in regular_df.columns:
        print("Averages already calculated! Skipping...")
        return regular_df, tourney_df
    
    # 1. Get unique games and enforce chronological order!
    unique_games = regular_df[regular_df['win'] == 1].sort_values(by=['Season', 'DayNum'])
    
    pre_game_elos = []
    final_elos = []
    
    for season, season_games in unique_games.groupby('Season'):
        print(season, end=' ')
        elo = {}
        for _, row in season_games.iterrows():
            w = row['T1_TeamID']
            l = row['T2_TeamID']
            day = row['DayNum']
            
            # THE CHANGE: Grab the Elo BEFORE the game starts
            ew = elo.get(w, initial)
            el = elo.get(l, initial)
            
            # Record these pre-game Elos so we can map them back to regular_df
            pre_game_elos.append({
                'Season': season, 'DayNum': day, 'WTeamID': w, 'W_pre_elo': ew, 'LTeamID': l, 'L_pre_elo': el
            })
            
            # Now do the Elo Math to update the dictionary for the NEXT game
            expected_w = 1.0 / (1.0 + 10 ** ((el - ew) / 400.0))
            elo[w] = ew + k * (1.0 - expected_w)
            elo[l] = el + k * (0.0 - (1.0 - expected_w))
            
        # At the end of the season, save the final Elos for the Tournament
        for team_id, rating in elo.items():
            final_elos.append({'Season': season, 'TeamID': team_id, 'elo': rating})
            
    # 2. Convert our records into DataFrames
    pre_game_df = pd.DataFrame(pre_game_elos)
    final_elo_df = pd.DataFrame(final_elos)
    
    # 3. Restructure pre-game Elos to easily merge onto the DOUBLED regular_df
    # We break it into a simple Season-DayNum-TeamID mapping
    w_elo = pre_game_df[['Season', 'DayNum', 'WTeamID', 'W_pre_elo']].rename(
        columns={'WTeamID': 'TeamID', 'W_pre_elo': 'elo'}
    )
    l_elo = pre_game_df[['Season', 'DayNum', 'LTeamID', 'L_pre_elo']].rename(
        columns={'LTeamID': 'TeamID', 'L_pre_elo': 'elo'}
    )
    team_pre_elos = pd.concat([w_elo, l_elo], ignore_index=True)
    
    # 4. Create T1 and T2 mapping tables
    t1_mapping = team_pre_elos.copy()
    t1_mapping.columns = ['Season', 'DayNum', 'T1_TeamID', 'T1_elo']
    
    t2_mapping = team_pre_elos.copy()
    t2_mapping.columns = ['Season', 'DayNum', 'T2_TeamID', 'T2_elo']

    # 6. Merge Pre-Game Elos onto Regular Season Data
    regular_df = pd.merge(regular_df, t1_mapping, on=['Season', 'DayNum', 'T1_TeamID'], how='left')
    regular_df = pd.merge(regular_df, t2_mapping, on=['Season', 'DayNum', 'T2_TeamID'], how='left')
    
    # 7. Merge Final Season Elos onto Tournament Data
    t1_final = final_elo_df.copy()
    t1_final.columns = ['Season', 'T1_TeamID', 'T1_elo']
    
    t2_final = final_elo_df.copy()
    t2_final.columns = ['Season', 'T2_TeamID', 'T2_elo']
    
    tourney_df = pd.merge(tourney_df, t1_final, on=['Season', 'T1_TeamID'], how='left')
    tourney_df = pd.merge(tourney_df, t2_final, on=['Season', 'T2_TeamID'], how='left')
    
    # 8. Safely fill missing values
    regular_df['T1_elo'] = regular_df['T1_elo'].fillna(initial)
    regular_df['T2_elo'] = regular_df['T2_elo'].fillna(initial)
    tourney_df['T1_elo'] = tourney_df['T1_elo'].fillna(initial)
    tourney_df['T2_elo'] = tourney_df['T2_elo'].fillna(initial)
    
    print("Done!")
    return regular_df, tourney_df

# Apply the function
regular_results_detailed, tourney_results_detailed = compute_rolling_elo(regular_results_detailed, tourney_results_detailed)

Calculating rolling Elo ratings...
2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026 Done!


MODEL PREP

In [20]:
def prepare_tourney_focused_split(regular_df:pd.DataFrame, tourney_df:pd.DataFrame , val_start_season, test_season):
    # 1. Identify valid numeric features
    exclude_cols = ['Season', 'DayNum', 'T1_TeamID', 'T2_TeamID', 'T1_Loc', 'T2_Loc', 
                    'PointDiff', 'win', 'T1_seed', 'T2_seed', 'men_women']
    
    t1_cols = [col for col in regular_df.columns if col.startswith('T1_')]
    core_features = [col.replace('T1_', '') for col in t1_cols if col not in exclude_cols]
    
    def calculate_differences(df:pd.DataFrame):
        diff_df = pd.DataFrame()
        diff_df['Season'] = df['Season']
        diff_df['win'] = df['win']
        diff_df['Seed_diff'] = df['Seed_diff']
        diff_df['men_women'] = df['men_women']
        
        for feature in core_features:
            if f'T1_{feature}' in df.columns and f'T2_{feature}' in df.columns:
                diff_df[f'diff_{feature}'] = df[f'T1_{feature}'] - df[f'T2_{feature}']
        return diff_df

    # Process differences separately before mixing
    reg_diffs = calculate_differences(regular_df)
    tourney_diffs = calculate_differences(tourney_df)
    
    feature_cols = [col for col in reg_diffs.columns if col not in ['Season', 'win']]
    
    # 2. THE NEW SPLIT LOGIC
    
    # TRAIN: All regular season games before test_season + older tournament games (before val_season)
    #2026 has filled in reg, but not tourney
    if test_season == 2026:
        train_reg = reg_diffs[reg_diffs['Season'] <= test_season]
    else:
        train_reg = reg_diffs[reg_diffs['Season'] < test_season]
    train_tourney = tourney_diffs[tourney_diffs['Season'] < val_start_season]
    train_df = pd.concat([train_reg, train_tourney], axis=0, ignore_index=True)
    
    # VAL: ONLY Tournament games from recent seasons
    # This ensures early stopping optimizes for the tournament!
    val_df = tourney_diffs[(tourney_diffs['Season'] >= val_start_season) & 
                           (tourney_diffs['Season'] < test_season)].copy()
    
    # TEST: 2025 as the holdout season to determine final quality of approach
    #empty for 2026 as we are sure of the network and want to feed it all the
    test_df = tourney_diffs[tourney_diffs['Season'] == test_season].copy()
    
    # 3. Scale the Data (Fit strictly on TRAIN)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df[feature_cols])
    X_val_scaled = scaler.transform(val_df[feature_cols])
    
    # Handle the empty test set gracefully
    if len(test_df) > 0:
        X_test_scaled = scaler.transform(test_df[feature_cols])
    else:
        X_test_scaled = np.array([])
    
    # 4. Create PyTorch DataLoaders
    def to_loader(X:np.ndarray, y:pd.Series, batch_size, shuffle=False):
        if len(X) == 0:
            return [] # Return empty list if no data (e.g., 2026 test set)
        tensor_X = torch.FloatTensor(X)
        tensor_y = torch.FloatTensor(y.values).unsqueeze(1)
        return DataLoader(TensorDataset(tensor_X, tensor_y), batch_size=batch_size, shuffle=shuffle)
    
    loaders = {
        'train': to_loader(X_train_scaled, train_df['win'], batch_size=512, shuffle=True),
        'val': to_loader(X_val_scaled, val_df['win'], batch_size=64, shuffle=False),
        'test': to_loader(X_test_scaled, test_df['win'] if len(test_df) > 0 else pd.Series([]), batch_size=64, shuffle=False),
        'input_dim': len(feature_cols),
        'scaler': scaler 
    }
    
    print(f"Train Shape (Reg + Old Tourney): {X_train_scaled.shape}")
    print(f"Val Shape ({val_start_season}-{test_season-1} Tourney Only): {X_val_scaled.shape}")
    print(f"Test Shape ({test_season} Tourney Only): {X_test_scaled.shape}")
    
    return loaders

# Execute
data_splits = prepare_tourney_focused_split(regular_results_detailed, tourney_results_detailed, val_start_season=CFG['val_season'], test_season=2025)

Train Shape (Reg + Old Tourney): (382500, 18)
Val Shape (2021-2024 Tourney Only): (1062, 18)
Test Shape (2025 Tourney Only): (268, 18)


# MODEL

In [21]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),     
            nn.ReLU(),
            nn.Dropout(CFG['dropout']),        
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(CFG['dropout']),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)

# TRAIN

In [22]:
def train_pass(data_splits: dict):
    model = MLP(input_dim=data_splits['input_dim']).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss() 
    optimizer = Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

    best_val_brier = float('inf') 
    epochs_no_improve = 0
    best_model_state = None

    print("Starting Single-Run Training...")
    print("-" * 50)

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0.0
        
        # --- Training Pass ---
        for batch_X, batch_y in data_splits['train']:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            
            # Apply Label Smoothing to targets
            smoothed_y = batch_y * (1.0 - CFG['label_smoothing']) + 0.5 * CFG['label_smoothing']
            
            optimizer.zero_grad()
            predictions:torch.Tensor = model(batch_X) #forward pass
            loss:torch.Tensor = criterion(predictions, smoothed_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        # --- Validation Pass ---
        model.eval()
        val_loss = 0.0
        val_brier = 0.0           
        total_val_samples = 0     
        
        with torch.no_grad():
            for batch_X, batch_y in data_splits['val']:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                predictions:torch.Tensor = model(batch_X)
                
                # Standard BCE Loss for information
                loss:torch.Tensor = criterion(predictions, batch_y) 
                val_loss += loss.item()
                
                # Brier Score Calculation (MSE of probabilities)
                probs = torch.sigmoid(predictions)
                val_brier += torch.sum((probs - batch_y) ** 2).item()
                total_val_samples += batch_y.size(0)
                
        # Calculate Epoch Averages
        train_loss /= len(data_splits['train'])
        val_loss /= len(data_splits['val'])
        val_brier /= total_val_samples 
        
        if epoch % 10 == 0 or DEVICE.type in ('xpu', 'cpu'):
            print(f"Epoch {epoch:3d} | Train Loss (BCE): {train_loss:.4f} | Val Loss (BCE): {val_loss:.4f} | Val Brier: {val_brier:.4f}")
        
        # --- Early Stopping (Optimizing for Brier) ---
        if val_brier < best_val_brier:
            best_val_brier = val_brier
            epochs_no_improve = 0
            # THE FIX: Deep clone the tensors so they detach from the actively training model
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CFG['patience']:
                print(f"Early stopping triggered at epoch {epoch}. Best Val Brier: {best_val_brier:.4f}")
                break

    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model
        
model = train_pass(data_splits)
print("-" * 50)
print("Training Complete. Model loaded with best validation weights.")

Starting Single-Run Training...
--------------------------------------------------
Epoch   0 | Train Loss (BCE): 0.5796 | Val Loss (BCE): 0.5417 | Val Brier: 0.1832
Epoch  10 | Train Loss (BCE): 0.5670 | Val Loss (BCE): 0.5398 | Val Brier: 0.1816
Epoch  20 | Train Loss (BCE): 0.5666 | Val Loss (BCE): 0.5366 | Val Brier: 0.1810
Epoch  30 | Train Loss (BCE): 0.5669 | Val Loss (BCE): 0.5375 | Val Brier: 0.1813
Epoch  40 | Train Loss (BCE): 0.5665 | Val Loss (BCE): 0.5395 | Val Brier: 0.1816
Early stopping triggered at epoch 44. Best Val Brier: 0.1808
--------------------------------------------------
Training Complete. Model loaded with best validation weights.


# GENERATE SUB

In [23]:
def generate_submission(model, scaler, regular_df:pd.DataFrame, sample_sub_df:pd.DataFrame):
    print("Preparing Submission Data...")
    sub = sample_sub_df.copy()
    
    # 1. Parse the Kaggle ID string
    sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
    sub['T1_TeamID'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
    sub['T2_TeamID'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))
    
    # 2. Add men_women flag (1 if ID starts with 1, else 0)
    sub['men_women'] = sub['T1_TeamID'].apply(lambda x: 1 if str(x).startswith('1') else 0)
    
    # 3. Add Seeds (using the function we fixed earlier with the ghost seed)
    sub = seed_transform(sub)
    
    # 4. Extract the FINAL end-of-season rolling stats for the tournament
    feature_cols = [c for c in regular_df.columns if c.startswith('T1_avg_')]
    if 'T1_elo' in regular_df.columns:
        feature_cols.append('T1_elo')
        
    temp_df = regular_df.sort_values(by=['Season', 'DayNum'])
    season_averages = temp_df[['Season', 'T1_TeamID'] + feature_cols].drop_duplicates(subset=['Season', 'T1_TeamID'], keep='last')
    
    # 5. Merge averages for T1
    sub = pd.merge(sub, season_averages, on=['Season', 'T1_TeamID'], how='left')
    
    # 6. Create T2 averages mapping and merge for T2
    season_averages_t2 = season_averages.copy()
    # THE FIX: Use feature_cols and just replace 'T1_' with 'T2_' so it catches both averages AND Elo!
    season_averages_t2.columns = ['Season', 'T2_TeamID'] + [c.replace('T1_', 'T2_') for c in feature_cols]
    sub = pd.merge(sub, season_averages_t2, on=['Season', 'T2_TeamID'], how='left')
    
    # 7. DYNAMIC DIFFERENCE CALCULATION
    # Read exactly what the scaler was trained on to prevent KeyErrors
    feature_cols = list(scaler.feature_names_in_)
    
    for col in feature_cols:
        if col.startswith('diff_'):
            base_feat = col.replace('diff_', '') # e.g., 'avg_Score'
            t1_col = f'T1_{base_feat}'           # e.g., 'T1_avg_Score'
            t2_col = f'T2_{base_feat}'           # e.g., 'T2_avg_Score'
            sub[col] = sub[t1_col] - sub[t2_col]
            
    # Check for missing averages (happens if a team had no D1 regular season games)
    if sub[feature_cols].isnull().any().any():
        print("Warning: Missing averages found in submission set. Filling with 0.")
        sub[feature_cols] = sub[feature_cols].fillna(0)
        
    # 8. Scale the features using the fitted training scaler
    X_sub_scaled = scaler.transform(sub[feature_cols])
    
    # 9. Predict using the neural network
    print("Running neural network predictions...")
    model.eval()
    tensor_X = torch.FloatTensor(X_sub_scaled).to(DEVICE)
    
    with torch.no_grad():
        predictions = model(tensor_X)
        probs = torch.sigmoid(predictions).cpu().numpy().squeeze()
        
    # 10. Clip probabilities to avoid infinite log loss penalties
    sub["Pred"] = np.clip(probs, 1e-5, 1 - 1e-5)
    
    # 11. Format final output
    submission = sub[["ID", "Pred"]]
    submission.to_csv("submission.csv", index=False)
    
    print(f"Success! Submission saved to 'submission.csv'. Shape: {submission.shape}")
    return submission

In [24]:
# ==============================================================================
# FINAL EXECUTION BLOCK: DATA PREP AND SUBMISSION
# ==============================================================================
if CFG['finalize']:
    # --- EVALUATE ON 2025 HOLDOUT ---
    model.eval()
    test_brier = 0.0
    total_test_samples = 0

    with torch.no_grad():
        for batch_X, batch_y in data_splits['test']:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            
            predictions = model(batch_X)

            probs = torch.sigmoid(predictions)
            test_brier += torch.sum((probs - batch_y) ** 2).item()
            total_test_samples += batch_y.size(0)

    test_brier /= total_test_samples

    print("=" * 50)
    print(f"đźŹ€ FINAL HOLDOUT SCORE (2025 Tourney) Brier: {test_brier:.4f} đźŹ€")
    print("=" * 50)

    print("1. Preparing Production Data for 2026...")
    data_splits_prod = prepare_tourney_focused_split(
        regular_results_detailed, 
        tourney_results_detailed, 
        val_start_season=CFG['val_season'], 
        test_season=2026 
    )

    #potential seed ensemble
    random_seeds = [CFG['seed']]
    all_preds = []
    criterion = nn.BCEWithLogitsLoss() 

    for s in random_seeds:
        print(f"\n{'='*40}")
        print(f"--- Training Production Model with Seed {s} ---")
        print(f"{'='*40}")
        
        # 2. Lock in the seed for this specific run
        torch.manual_seed(s)
        np.random.seed(s)

        final_model = train_pass(data_splits_prod)
            
        # 6. Generate predictions for SampleSubmissionStage2
        print(f"\nGenerating Stage 2 predictions for Seed {s}...")
        sub = generate_submission(
            model=final_model, 
            scaler=data_splits_prod['scaler'], 
            regular_df=regular_results_detailed, 
            sample_sub_df=DATA['sample_sub2']
        )
        
        # Store the predicted probabilities
        all_preds.append(sub['Pred'].values)


    # 7. Finalize the Ensemble
    print("\n" + "=" * 50)
    print("đźŹ€ Averaging predictions for final ensemble... đźŹ€")
    final_submission = DATA['sample_sub2'].copy()
    final_submission['Pred'] = np.mean(all_preds, axis=0)

    # Save the ultimate file
    final_submission.to_csv("submission.csv", index=False)
    print("Success! Ensembled submission saved to 'submission.csv'.")
    print("=" * 50)

đźŹ€ FINAL HOLDOUT SCORE (2025 Tourney) Brier: 0.1373 đźŹ€
1. Preparing Production Data for 2026...
Train Shape (Reg + Old Tourney): (426945, 18)
Val Shape (2021-2025 Tourney Only): (1330, 18)
Test Shape (2026 Tourney Only): (0,)

--- Training Production Model with Seed 923434 ---
Starting Single-Run Training...
--------------------------------------------------
Epoch   0 | Train Loss (BCE): 0.5771 | Val Loss (BCE): 0.5224 | Val Brier: 0.1746
Epoch  10 | Train Loss (BCE): 0.5653 | Val Loss (BCE): 0.5202 | Val Brier: 0.1731
Epoch  20 | Train Loss (BCE): 0.5649 | Val Loss (BCE): 0.5173 | Val Brier: 0.1722
Epoch  30 | Train Loss (BCE): 0.5652 | Val Loss (BCE): 0.5160 | Val Brier: 0.1719
Early stopping triggered at epoch 37. Best Val Brier: 0.1716

Generating Stage 2 predictions for Seed 923434...
Preparing Submission Data...
Running neural network predictions...
Success! Submission saved to 'submission.csv'. Shape: (132133, 2)

đźŹ€ Averaging predictions for final ensemble... đźŹ€
Success